# Petulis Search — Notebook completo

Motor de búsqueda distribuido sobre artículos de Wikipedia usando **Apache Spark**, **MapReduce** y **TF-IDF**.

Este notebook reúne en un solo flujo lo que normalmente estaría separado en:

- `src/common.py`
- `src/build_index.py`
- `src/search.py`

La lógica se mantiene igual, pero se adapta a Jupyter Notebook. Por eso se eliminan `argparse`, `__file__`, `sys.path.append(...)` y `if __name__ == "__main__"`, porque en notebook los parámetros se definen directamente en celdas.

## 1. Requisitos

Según el README del proyecto, el entorno recomendado es WSL con Ubuntu, Python 3, Java 17, entorno virtual y PySpark.

Comandos sugeridos en WSL:

```bash
sudo apt update
sudo apt upgrade -y
sudo apt install -y python3 python3-pip python3-venv
sudo apt install -y openjdk-17-jdk
python3 -m venv .venv
source .venv/bin/activate
pip install pyspark jupyter
```

En caso de estar ejecutando este notebook desde un entorno donde PySpark no esté instalado, se puede ejecutar la siguiente celda.

In [ ]:
# Ejecutar solo si PySpark no está instalado en el entorno actual.
# %pip install pyspark

## 2. Estructura esperada del proyecto

El notebook asume la misma estructura indicada en el README:

```text
petulis-search/
│
├── README.md
├── src/
│   ├── common.py
│   ├── build_index.py
│   └── search.py
│
├── data/
│   ├── raw/
│   │   └── archivos_json_de_wikipedia/
│   │
│   ├── sample/
│   │   └── muestra_de_archivos_json/
│   │
│   └── index/
│
└── conf/
    └── log4j2.properties
```

Para este notebook, lo importante es que los archivos JSON de Wikipedia estén dentro de:

```text
data/raw/
```

Y que el índice se guarde en:

```text
data/index/
```

## 3. Dataset esperado

El dataset usado por el proyecto es **Plain Text Wikipedia 2020-11**. Cada archivo contiene artículos en JSON con una estructura similar a:

```json
{
  "id": "7751000",
  "title": "M-137 (Michigan highway)",
  "text": "M-137 was a state trunkline highway..."
}
```

Este notebook está configurado para leer archivos JSON multilineales, igual que el comando del README:

```bash
--format json \
--multiline-json \
--id-col id \
--title-col title \
--text-col text
```

## 4. Importaciones generales

In [ ]:
import re
import json
import math
import unicodedata
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
)

# Parte 1 — Código equivalente a `src/common.py`

Contiene las funciones comunes de preprocesamiento de texto:

- Convertir texto a minúsculas.
- Eliminar acentos.
- Extraer palabras válidas usando expresiones regulares.
- Eliminar stopwords.
- Tokenizar documentos y consultas.

In [ ]:
# Conjunto de palabras muy frecuentes en inglés que
# no aportan significado relevante en una búsqueda.
STOPWORDS = {
    "the", "and", "for", "are", "with", "that", "this", "from", "was", "were",
    "has", "have", "had", "not", "but", "you", "your", "its", "his", "her",
    "they", "them", "their", "into", "about", "than", "then", "also", "been",
    "can", "may", "such", "when", "where", "which", "while", "who", "what",
    "how", "why", "all", "any", "one", "two", "more", "most", "other",
    "some", "each", "many", "much", "very", "will", "would", "could",
    "should", "there", "these", "those", "between", "because", "through",
    "over", "under", "after", "before", "during", "within", "without",
    "is", "a", "an", "of", "to", "in", "on", "by", "as", "at", "or",
    "it", "be", "if"
}

# Expresión regular usada para identificar términos válidos.
#
# Patrón:
#
# [a-zA-Z]
#     La palabra debe comenzar con una letra.
#
# [a-zA-Z0-9_]{2,}
#     Debe continuar con al menos dos caracteres más.
TOKEN_RE = re.compile(r"[a-zA-Z][a-zA-Z0-9_]{2,}")


def normalize_text(text: str) -> str:
    """
    Normaliza el texto antes de tokenizarlo.

    Ejemplos:
    "Artificial Intelligence"
        -> "artificial intelligence"
    """

    if text is None:
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFD", text)

    text = "".join(
        ch
        for ch in text
        if unicodedata.category(ch) != "Mn"
    )

    return text


def tokenize(text: str) -> list[str]:
    """
    Función principal de preprocesamiento de texto.

    Esta función transforma un documento de texto libre
    en una lista de términos que podrán ser indexados.

    Ejemplo:

    Entrada:
        "Artificial Intelligence and Machine Learning"

    Después de normalizar:
        "artificial intelligence and machine learning"

    Tokens encontrados:
        [
            "artificial",
            "intelligence",
            "and",
            "machine",
            "learning"
        ]

    Después de eliminar stopwords:
        [
            "artificial",
            "intelligence",
            "machine",
            "learning"
        ]
    """

    # Normalizar el texto.
    text = normalize_text(text)

    # Extraer tokens usando la expresión regular.
    tokens = TOKEN_RE.findall(text)

    clean_tokens = []

    # Filtrar tokens irrelevantes.
    for token in tokens:

        # Eliminar stopwords.
        if token not in STOPWORDS and len(token) >= 3:
            clean_tokens.append(token)

    return clean_tokens

## Prueba rápida del preprocesamiento

In [ ]:
tokenize("Artificial Intelligence and Machine Learning")

# Parte 2 — Crear sesión de Spark

En el README se ejecuta con `spark-submit --master "local[*]"`. En notebook, se replica esa idea usando `.master("local[*]")`.

In [ ]:
def create_spark(app_name: str = "PetulisSearchNotebook", shuffle_partitions: int = 8) -> SparkSession:
    """
    Crea la sesión de Spark.

    SparkSession es el punto de entrada para usar Spark. Se configura:
    - Nombre de la aplicación.
    - Modo local usando todos los núcleos disponibles.
    - Ocultar barra de progreso.
    - Número de particiones para operaciones de shuffle.
    """

    builder = (
        SparkSession.builder
        .master("local[*]")
        .appName(app_name)
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.shuffle.partitions", str(shuffle_partitions))
    )

    # Si existe el archivo de configuración de logs indicado en el README,
    # se usa para reducir mensajes internos de Spark.
    log4j_path = Path("conf/log4j2.properties")
    if log4j_path.exists():
        builder = builder.config(
            "spark.driver.extraJavaOptions",
            "-Dlog4j.configurationFile=conf/log4j2.properties"
        )

    return builder.getOrCreate()


spark = create_spark(shuffle_partitions=8)
spark.sparkContext.setLogLevel("WARN")

# Parte 3 — Parámetros generales del notebook

Estos valores equivalen a los argumentos usados en los comandos del README.

Crear índice en terminal:

```bash
spark-submit --master "local[*]" \
  --py-files src/common.py \
  src/build_index.py \
  --input data/raw/ \
  --output data/index/ \
  --format json \
  --multiline-json \
  --id-col id \
  --title-col title \
  --text-col text \
  --partitions 8 \
  --min-df 2 \
  --overwrite
```

En notebook, esos argumentos se convierten en variables.

In [ ]:
# Ruta donde están los archivos JSON de Wikipedia.
INPUT_PATH = "data/raw/"

# Ruta donde se guardará el índice invertido.
OUTPUT_INDEX_PATH = "data/index/"

# Formato del dataset según el README.
DATASET_FORMAT = "json"
MULTILINE_JSON = True

# Columnas del dataset.
ID_COL = "id"
TITLE_COL = "title"
TEXT_COL = "text"

# Parámetros de procesamiento distribuido.
PARTITIONS = 8
MIN_DF = 2
MAX_DF_RATIO = 0.80

# 0 significa procesar todos los documentos.
# Para pruebas rápidas se puede usar, por ejemplo, 1000.
MAX_DOCS = 0

# Sobrescribir el índice si ya existe.
OVERWRITE = True

# Parte 4 — Código equivalente a `src/build_index.py`

Esta sección construye el índice invertido TF-IDF.

Realiza:

1. Lectura de artículos de Wikipedia.
2. Limpieza y tokenización.
3. Cálculo de TF.
4. Cálculo de DF.
5. Cálculo de IDF.
6. Cálculo de TF-IDF.
7. Construcción del índice invertido.
8. Guardado del índice en disco.

In [ ]:
def pick_column(columns, preferred, candidates):
    """
    Escoge una columna del dataset.

    Primero intenta usar la columna indicada por el usuario.
    Si el usuario no indicó una, busca entre nombres comunes.

    Ejemplo:
    Para texto puede buscar:
    text, content, body, article, document.
    """

    if preferred:
        if preferred not in columns:
            raise ValueError(f"La columna indicada no existe: {preferred}")
        return preferred

    for candidate in candidates:
        if candidate in columns:
            return candidate

    return None


def load_documents_df(spark: SparkSession, args):
    """
    Carga los documentos desde el dataset.

    El resultado final siempre tendrá esta forma:

    doc_id | title | text
    """

    reader = spark.read.option("recursiveFileLookup", "true")

    # Si el JSON es multilineal, entonces Spark necesita multiLine=true.
    if args.format == "json":
        reader = reader.option("multiLine", str(args.multiline_json).lower())

    df = reader.json(args.input)

    # Permite inspeccionar columnas antes de procesar.
    if args.show_schema:
        df.printSchema()
        print("Columnas detectadas:", df.columns)
        return None

    columns = df.columns

    id_col = pick_column(
        columns,
        args.id_col,
        ["id", "doc_id", "page_id", "article_id", "url"],
    )

    title_col = pick_column(
        columns,
        args.title_col,
        ["title", "name", "article_title"],
    )

    text_col = pick_column(
        columns,
        args.text_col,
        ["text", "content", "body", "article", "document"],
    )

    if text_col is None:
        raise ValueError(
            "No pude detectar la columna de texto. "
            "Ejecutá con show_schema=True y luego indicá text_col."
        )

    # Si no existe columna de ID, Spark genera una.
    doc_id_expr = (
        F.col(id_col).cast("string")
        if id_col
        else F.monotonically_increasing_id().cast("string")
    )

    # Si no existe título, se genera uno básico.
    title_expr = (
        F.col(title_col).cast("string")
        if title_col
        else F.concat(F.lit("doc-"), doc_id_expr)
    )

    docs = df.select(
        doc_id_expr.alias("doc_id"),
        title_expr.alias("title"),
        F.col(text_col).cast("string").alias("text"),
    )

    docs = docs.filter(F.col("text").isNotNull())

    if args.max_docs > 0:
        docs = docs.limit(args.max_docs)

    return docs


def document_to_tf_pairs(row):
    """
    MAP principal del proyecto.

    Entrada:
        Un documento con:
        - doc_id
        - title
        - text

    Salida:
        Lista de pares:

        ((term, doc_id), tf)

    Ejemplo:

        Documento doc1:
        "machine learning machine"

        Tokens:
        ["machine", "learning", "machine"]

        Frecuencias:
        machine = 2
        learning = 1

        TF:
        machine = 2 / 3
        learning = 1 / 3

        Salida:
        (("machine", "doc1"), 0.666)
        (("learning", "doc1"), 0.333)
    """

    doc_id = row["doc_id"]
    tokens = tokenize(row["text"])

    if not tokens:
        return []

    counts = Counter(tokens)
    total_terms = len(tokens)

    return [
        ((term, doc_id), count / total_terms)
        for term, count in counts.items()
    ]

In [ ]:
def build_index(spark: SparkSession, args):
    """
    Construye el índice invertido TF-IDF.
    """

    spark.sparkContext.setLogLevel("ERROR")

    output_path = Path(args.output)
    mode = "overwrite" if args.overwrite else "error"

    # Carga documentos y los reparte en particiones.
    docs_df = load_documents_df(spark, args)

    if docs_df is None:
        return None

    docs_df = docs_df.repartition(args.partitions)
    docs_df.cache()

    total_docs = docs_df.count()

    if total_docs == 0:
        raise ValueError("No se encontraron documentos para procesar.")

    print(f"Documentos procesados: {total_docs}")
    print(f"Particiones usadas: {docs_df.rdd.getNumPartitions()}")

    # Guarda únicamente la metadata de documentos.
    # Esto permite mostrar títulos en las búsquedas.
    docs_meta_df = docs_df.select("doc_id", "title")

    # MAP + REDUCE 1: cálculo de TF
    #
    # MAP:
    # documento -> ((term, doc_id), tf)
    #
    # REDUCE:
    # agrupa por (term, doc_id)
    term_doc_tf_rdd = (
        docs_df.rdd
        .flatMap(document_to_tf_pairs)
        .reduceByKey(lambda a, b: a + b, numPartitions=args.partitions)
    )

    # REDUCE 2: cálculo de DF
    #
    # DF = Document Frequency.
    #
    # Indica en cuántos documentos aparece cada término.
    #
    # Entrada:
    # ((term, doc_id), tf)
    #
    # Salida:
    # (term, cantidad_de_documentos)
    term_df_rdd = (
        term_doc_tf_rdd
        .map(lambda item: (item[0][0], 1))
        .reduceByKey(lambda a, b: a + b, numPartitions=args.partitions)
    )

    # Se descartan términos demasiado comunes.
    # Por ejemplo, si un término aparece en más del 80% de documentos,
    # aporta poca relevancia para diferenciar resultados.
    max_df = int(total_docs * args.max_df_ratio)

    filtered_df_rdd = term_df_rdd.filter(
        lambda item: args.min_df <= item[1] <= max_df
    )

    # Cálculo de IDF
    #
    # Fórmula:
    # idf = log((N + 1) / (df + 1)) + 1
    #
    # N  = total de documentos
    # df = cantidad de documentos donde aparece el término
    term_idf_rdd = filtered_df_rdd.mapValues(
        lambda df: math.log((total_docs + 1) / (df + 1)) + 1
    )

    # Join entre TF e IDF
    #
    # Se reorganiza el RDD para que la clave sea el término.
    #
    # Antes:
    # ((term, doc_id), tf)
    #
    # Después:
    # (term, (doc_id, tf))
    tf_by_term_rdd = term_doc_tf_rdd.map(
        lambda item: (item[0][0], (item[0][1], item[1]))
    )

    # Se une cada TF con el IDF correspondiente.
    #
    # Resultado final:
    # term, doc_id, tf, idf, tfidf
    tfidf_rdd = (
        tf_by_term_rdd
        .join(term_idf_rdd, numPartitions=args.partitions)
        .map(
            lambda item: (
                item[0],                 # term
                item[1][0][0],           # doc_id
                float(item[1][0][1]),    # tf
                float(item[1][1]),       # idf
                float(item[1][0][1] * item[1][1]),  # tfidf
            )
        )
    )

    # Esquema del índice invertido.
    schema = StructType([
        StructField("term", StringType(), False),
        StructField("doc_id", StringType(), False),
        StructField("tf", DoubleType(), False),
        StructField("idf", DoubleType(), False),
        StructField("tfidf", DoubleType(), False),
    ])

    postings_df = spark.createDataFrame(tfidf_rdd, schema)

    # Bucket basado en la primera letra del término.
    #
    # Esto permite que search.py filtre por bucket
    # y no tenga que revisar todo el índice completo.
    postings_df = postings_df.withColumn(
        "bucket",
        F.substring(F.col("term"), 1, 1),
    )

    postings_output = str(output_path / "postings")
    docs_output = str(output_path / "documents")

    # Guarda el índice invertido.
    postings_df.write.mode(mode).json(postings_output)

    # Guarda metadata de documentos.
    docs_meta_df.write.mode(mode).json(docs_output)

    # Guarda estadísticas útiles para mostrar durante la búsqueda.
    stats = {
        "total_docs": total_docs,
        "partitions": args.partitions,
        "min_df": args.min_df,
        "max_df_ratio": args.max_df_ratio,
        "description": "Petulis Search TF-IDF inverted index",
    }

    output_path.mkdir(parents=True, exist_ok=True)

    with open(output_path / "stats.json", "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=4)

    print("Índice construido correctamente.")
    print(f"Postings: {postings_output}")
    print(f"Documentos: {docs_output}")
    print(f"Stats: {output_path / 'stats.json'}")

    docs_df.unpersist()

    return {
        "total_docs": total_docs,
        "postings": postings_output,
        "documents": docs_output,
        "stats": str(output_path / "stats.json"),
    }

## 4.1 Revisar esquema del dataset

Antes de construir el índice completo, se puede revisar el esquema detectado por Spark.

Ejecutar esta celda es opcional, pero ayuda a confirmar que existen las columnas `id`, `title` y `text`.

In [ ]:
schema_args = SimpleNamespace(
    input=INPUT_PATH,
    output=OUTPUT_INDEX_PATH,
    format=DATASET_FORMAT,
    multiline_json=MULTILINE_JSON,
    id_col=ID_COL,
    title_col=TITLE_COL,
    text_col=TEXT_COL,
    max_docs=MAX_DOCS,
    partitions=PARTITIONS,
    min_df=MIN_DF,
    max_df_ratio=MAX_DF_RATIO,
    overwrite=OVERWRITE,
    show_schema=True,
)

# Descomentar para inspeccionar el esquema del dataset.
# load_documents_df(spark, schema_args)

## 4.2 Construir el índice

Esta celda reemplaza el comando `spark-submit` de construcción del índice. Usa los mismos valores indicados en el README:

- `input`: `data/raw/`
- `output`: `data/index/`
- `format`: `json`
- `multiline_json`: `True`
- `id_col`: `id`
- `title_col`: `title`
- `text_col`: `text`
- `partitions`: `8`
- `min_df`: `2`
- `overwrite`: `True`

> Nota: esta celda puede tardar dependiendo del tamaño del dataset.

In [ ]:
build_args = SimpleNamespace(
    input=INPUT_PATH,
    output=OUTPUT_INDEX_PATH,
    format=DATASET_FORMAT,
    multiline_json=MULTILINE_JSON,
    id_col=ID_COL,
    title_col=TITLE_COL,
    text_col=TEXT_COL,
    max_docs=MAX_DOCS,
    partitions=PARTITIONS,
    min_df=MIN_DF,
    max_df_ratio=MAX_DF_RATIO,
    overwrite=OVERWRITE,
    show_schema=False,
)

# Ejecutar para construir el índice.
# build_info = build_index(spark, build_args)
# build_info

# Parte 5 — Código equivalente a `src/search.py`

Esta sección permite consultar el índice previamente construido. No vuelve a procesar Wikipedia.

Realiza:

1. Lee el índice invertido.
2. Tokeniza la consulta del usuario.
3. Busca los términos en el índice.
4. Agrupa resultados por documento.
5. Calcula un score de relevancia.
6. Muestra los documentos más relevantes.

In [ ]:
def search_index(spark: SparkSession, args):
    """
    Consulta el índice TF-IDF de Petulis Search.
    """

    spark.sparkContext.setLogLevel("WARN")

    index_path = Path(args.index)

    # Ruta donde están los postings del índice invertido.
    #
    # Los postings son las entradas del índice:
    # término -> documento -> tfidf
    #
    # Ejemplo:
    # "intelligence" -> doc_id 123 -> tfidf 0.58
    postings_path = str(index_path / "postings")

    # Ruta donde está la información básica de documentos.
    documents_path = str(index_path / "documents")

    # Ruta del archivo de estadísticas del índice.
    stats_path = index_path / "stats.json"

    # Procesa la consulta del usuario usando la misma función tokenize
    # que se usó durante la construcción del índice.
    #
    # Ejemplo:
    # "Artificial Intelligence Machine Learning"
    #
    # se convierte en:
    # ["artificial", "intelligence", "machine", "learning"]
    query_terms = tokenize(args.query)

    # Elimina términos repetidos manteniendo el orden.
    query_terms = list(dict.fromkeys(query_terms))

    # Si después de limpiar la consulta no queda ningún término válido,
    # el programa termina.
    #
    # Esto puede pasar si el usuario escribe solo palabras vacías como:
    # "the and of"
    if not query_terms:
        print("La consulta no contiene términos válidos.")
        return None

    # Obtiene la primera letra de cada término.
    #
    # Esto se usa porque el índice tiene una columna llamada "bucket",
    # creada a partir de la primera letra del término.
    #
    # Ejemplo:
    # ["artificial", "intelligence", "machine", "learning"]
    #
    # buckets:
    # ["a", "i", "m", "l"]
    buckets = list({term[0] for term in query_terms if len(term) > 0})

    # Muestra información inicial de la búsqueda.
    print(f"Consulta original: {args.query}")
    print(f"Términos usados: {query_terms}")

    # Si existe el archivo stats.json, lo lee y muestra cuántos documentos
    # fueron indexados.
    if stats_path.exists():
        with open(stats_path, "r", encoding="utf-8") as f:
            stats = json.load(f)

        print(f"Documentos en el índice: {stats.get('total_docs')}")

    # Lee el índice invertido desde disco.
    postings_df = spark.read.json(postings_path)

    # Lee la información de documentos.
    docs_df = spark.read.json(documents_path)

    # Filtra el índice para quedarse solamente con:
    # 1. Los buckets relacionados con la consulta.
    # 2. Los términos exactos de la consulta.
    #
    # Ejemplo:
    # Si la consulta tiene:
    # ["artificial", "intelligence"]
    #
    # solo se buscan términos:
    # artificial
    # intelligence
    #
    # y buckets:
    # a
    # i
    filtered = postings_df.filter(
        (F.col("bucket").isin(buckets)) &
        (F.col("term").isin(query_terms))
    )

    # Se calcula el ranking de documentos.
    ranked = (
        filtered

        # Agrupa todas las coincidencias por documento.
        .groupBy("doc_id")

        # Calcula dos cosas por documento:
        #
        # 1. score:
        #    suma de los valores TF-IDF de los términos encontrados.
        #
        # 2. matched_terms:
        #    conjunto de términos de la consulta que aparecieron
        #    en ese documento.
        .agg(
            F.sum("tfidf").alias("score"),
            F.collect_set("term").alias("matched_terms")
        )

        # Une el ranking con la tabla de documentos para obtener el título.
        .join(docs_df, on="doc_id", how="left")

        # Ordena los documentos de mayor a menor score.
        # Los documentos con mayor TF-IDF aparecen primero.
        .orderBy(F.desc("score"))

        # Limita la cantidad de resultados al valor indicado por top.
        .limit(args.top)
    )

    # collect() trae los resultados desde Spark hacia Python.
    results = ranked.collect()

    # Si no hay resultados, se informa al usuario.
    if not results:
        print("No se encontraron resultados.")

    else:
        print("\nResultados:")

        # Recorre los resultados y los imprime uno por uno.
        for i, row in enumerate(results, start=1):
            print("-" * 80)

            print(f"{i}. {row['title']}")

            print(f"   doc_id: {row['doc_id']}")

            # Score de relevancia.
            #
            # Este score es la suma de TF-IDF de los términos encontrados.
            # No es porcentaje ni probabilidad.
            # Mientras más alto, más relevante según el modelo.
            print(f"   score: {row['score']:.6f}")

            print(f"   términos encontrados: {', '.join(row['matched_terms'])}")

    return ranked

## 5.1 Realizar una búsqueda

Esta celda reemplaza el comando del README:

```bash
spark-submit --master "local[*]" \
  --py-files src/common.py \
  src/search.py \
  --index data/index/ \
  --query "artificial intelligence machine learning" \
  --top 10
```

In [ ]:
search_args = SimpleNamespace(
    index=OUTPUT_INDEX_PATH,
    query="artificial intelligence machine learning",
    top=10,
)

# Ejecutar después de construir el índice.
# ranked = search_index(spark, search_args)

## 5.2 Mostrar resultados como tabla en Jupyter

Además de imprimir los resultados como en terminal, en notebook se pueden mostrar como tabla.

In [ ]:
# Ejecutar después de realizar una búsqueda.
# ranked.select("title", "doc_id", "score", "matched_terms").show(truncate=False)

# Alternativa visual con Pandas:
# ranked.select("title", "doc_id", "score", "matched_terms").toPandas()

## 5.3 Otros ejemplos de búsqueda

Los ejemplos corresponden a los indicados en el README.

In [ ]:
# Ejemplo 1
# search_args.query = "distributed systems mapreduce"
# ranked = search_index(spark, search_args)

# Ejemplo 2
# search_args.query = "world war baseball aircraft"
# ranked = search_index(spark, search_args)

# Parte 6 — Interpretación del score

El score representa la suma de los valores TF-IDF de los términos encontrados en un documento.

No representa:

- Un porcentaje.
- Una probabilidad.

Un documento con score más alto se considera más relevante para la consulta según el modelo TF-IDF.

```text
score(documento) = suma de los TF-IDF de los términos encontrados
```

# Parte 7 — Relación con MapReduce

## Fase Map

Cada documento se transforma en pares clave-valor:

```text
((term, doc_id), tf)
```

En el código ocurre principalmente en:

```python
.flatMap(document_to_tf_pairs)
```

## Fase Reduce

Spark agrupa valores con la misma clave:

```python
.reduceByKey(lambda a, b: a + b)
```

Luego se calcula el DF:

```text
(term, cantidad_de_documentos)
```

Y finalmente se calcula el TF-IDF:

```text
TF-IDF = TF × IDF
```

# Parte 8 — Minimización del costo de comunicación

El proyecto reduce el costo de comunicación durante el procesamiento distribuido mediante:

- **Tokenización local:** cada documento se procesa en su partición.
- **Uso de `Counter`:** se cuentan palabras antes de emitir pares clave-valor.
- **Eliminación de stopwords:** se reducen términos frecuentes y poco relevantes.
- **Uso de `reduceByKey`:** reduce tráfico frente a `groupByKey`.
- **Filtro `min_df`:** elimina términos que aparecen en muy pocos documentos.
- **Filtro `max_df_ratio`:** elimina términos demasiado comunes.
- **Buckets:** permite que la búsqueda revise solo las particiones lógicas necesarias del índice.

# Parte 9 — Detener Spark

Cuando se termine de construir el índice y realizar búsquedas, se puede detener la sesión de Spark.

In [ ]:
# Ejecutar al final del trabajo.
# spark.stop()